# Notebook 18 — Mini-Project: compbio_utils v0

**Module:** 01 — Python & Scientific Computing  
**Notebook:** 18 of 20  
**Prerequisites:** Notebooks 01–17  
**Time estimate:** 3–4 hours (mini-project)

---
## Step 1 — Motivation

This notebook is the integration milestone for Module 01. The individual functions
written across Notebooks 02–17 are assembled into a coherent `compbio_utils v0`
package: tested, documented, packaged, and usable as a single-line import.

This package is a Track A and Track B portfolio artifact. A GitHub link to a
working, documented, tested biological utilities package is concrete evidence of
engineering competence.

---
## Step 2 — Intuition

A mini-project is not a new task — it's an integration of what was built in the
preceding notebooks. The work here is:

1. Verify the package is complete (all expected functions present)
2. Verify all tests pass
3. Verify `ruff` is clean
4. Write the `compbio_utils/README.md`
5. Tag `v0.1.0` in git
6. Run the expression analysis pipeline end-to-end using only `compbio_utils`

---
## Step 3 — Biological Background

**The end-to-end expression analysis pipeline:**

```
Raw counts matrix (genes × samples)
    ↓  clean_expression_matrix()
Filtered, deduplicated counts
    ↓  log2_cpm()
Normalized expression
    ↓  zscore_columns()
Scaled matrix
    ↓  pca_svd()             → pca_scatter() → portfolio figure
    ↓  log2_fold_change()    → volcano_plot() → portfolio figure
    ↓  expression_heatmap()                   → portfolio figure
```

This pipeline mirrors what is done in real published RNA-seq analyses. Running it
from scratch — using only functions you wrote — is the mini-project output.

---
## Step 4 — Mathematical Explanation

The pipeline applies three sequential normalizations:

1. **CPM**: removes library size variation between samples  
   $X^{\text{CPM}}_{ij} = X_{ij} / \sum_i X_{ij} \times 10^6$

2. **log2**: compresses dynamic range and makes distributions more normal  
   $X^{\log}_{ij} = \log_2(X^{\text{CPM}}_{ij} + 1)$

3. **z-score**: centers and scales each gene for cross-gene comparisons  
   $\hat{X}_{ij} = (X^{\log}_{ij} - \mu_j) / \sigma_j$

Each step removes a different source of technical variation.

---
## Step 5 — Computational Explanation

**Mini-project checklist:**

| Item | File | Status |
|------|------|--------|
| `gc_content`, `complement`, `reverse_complement`, `sliding_gc`, `gc_skew` | `sequence.py` | ☐ |
| `cpm_normalize`, `zscore_columns`, `pca_svd`, `bootstrap_ci`, `log2_fold_change` | `stats.py` | ☐ |
| `pca_scatter`, `volcano_plot`, `expression_heatmap` | `plotting.py` | ☐ |
| `read_fasta`, `iter_fasta`, `clean_expression_matrix` | `io.py` | ☐ |
| Test suite (all functions, ≥ 1 test each) | `tests/` | ☐ |
| `pyproject.toml` at v0.1.0 | `pyproject.toml` | ☐ |
| `README.md` | `README.md` | ☐ |
| `ruff check` clean | — | ☐ |
| All tests pass | — | ☐ |

---
## Step 6 — Python Implementation

In [ ]:
# Cell 6.1 — Full package audit
import pathlib, subprocess, sys

repo_root = pathlib.Path.cwd()
while repo_root != repo_root.parent:
    if (repo_root / ".git").exists(): break
    repo_root = repo_root.parent

cbu = repo_root / "utilities" / "compbio_utils"

REQUIRED = {
    "sequence.py": ["gc_content", "complement", "reverse_complement", "sliding_gc", "gc_skew"],
    "stats.py":    ["cpm_normalize", "zscore_columns", "pca_svd", "bootstrap_ci", "log2_fold_change"],
    "plotting.py": ["pca_scatter", "volcano_plot", "expression_heatmap"],
    "io.py":       ["read_fasta", "iter_fasta", "clean_expression_matrix"],
}

all_present = True
print("compbio_utils v0 audit:")
print("=" * 55)
for filename, funcs in REQUIRED.items():
    fpath = cbu / filename
    content = fpath.read_text() if fpath.exists() else ""
    print(f"\n  {filename}")
    for fn in funcs:
        ok = f"def {fn}" in content
        all_present = all_present and ok
        print(f"    {'✓' if ok else '✗ MISSING'}  {fn}")

print(f"\n{'All functions present.' if all_present else 'FIX: add missing functions before continuing.'}")

In [ ]:
# Cell 6.2 — Run test suite
test_result = subprocess.run(
    [sys.executable, "-m", "pytest", str(cbu / "tests"), "-v", "--tb=short"],
    capture_output=True, text=True, cwd=repo_root
)
print(test_result.stdout[-4000:])
tests_passed = test_result.returncode == 0
print(f"\nTests: {'PASS' if tests_passed else 'FAIL'}")

In [ ]:
# Cell 6.3 — Ruff lint check
ruff_result = subprocess.run(
    ["ruff", "check", str(cbu)],
    capture_output=True, text=True, cwd=repo_root
)
if ruff_result.returncode == 0:
    print("ruff: CLEAN")
else:
    print("ruff: issues found:")
    print(ruff_result.stdout)

In [ ]:
# Cell 6.4 — End-to-end pipeline using only compbio_utils
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(42)

# Simulate raw counts
n_genes, n_tumour, n_normal = 500, 6, 6
tumour_raw = rng.negative_binomial(n=8, p=0.3, size=(n_genes, n_tumour)).astype(float)
normal_raw = rng.negative_binomial(n=8, p=0.3, size=(n_genes, n_normal)).astype(float)

# Make 40 genes DE
tumour_raw[:20] *= 5
tumour_raw[20:40] /= 5

X_all = np.hstack([tumour_raw, normal_raw])
labels = ["tumour"]*n_tumour + ["normal"]*n_normal
gene_ids = [f"GENE_{i:04d}" for i in range(n_genes)]

# Try to use compbio_utils — fall back to inline if not installed
try:
    from compbio_utils.stats import cpm_normalize, zscore_columns, pca_svd, log2_fold_change
    from compbio_utils.plotting import pca_scatter, volcano_plot, expression_heatmap
    USING_CBU = True
    print("Using compbio_utils")
except ImportError:
    print("compbio_utils not available — using inline fallbacks (complete NB15 first)")
    USING_CBU = False

print(f"Raw matrix shape: {X_all.shape}")

In [ ]:
# Cell 6.5 — Run the pipeline
from scipy import stats as sp_stats

# Inline fallbacks (only used if compbio_utils missing)
if not USING_CBU:
    def cpm_normalize(X): return X / X.sum(axis=0) * 1e6
    def zscore_columns(X):
        mu = X.mean(axis=0); s = X.std(axis=0); s[s==0] = 1
        return (X - mu) / s
    def log2_fold_change(t, c, pseudocount=1):
        return np.log2(t.mean(axis=1)+pseudocount) - np.log2(c.mean(axis=1)+pseudocount)

# Step 1: CPM → log2
X_cpm  = cpm_normalize(X_all)
X_log  = np.log2(X_cpm + 1)

# Step 2: z-score
X_z    = zscore_columns(X_log)

# Step 3: fold change + p-values
lfc = log2_fold_change(tumour_raw, normal_raw, pseudocount=1)
pvals = np.array([
    sp_stats.ttest_ind(tumour_raw[i], normal_raw[i]).pvalue
    for i in range(n_genes)
])

print(f"DE genes (|lfc|>1, p<0.05): {((np.abs(lfc)>1) & (pvals<0.05)).sum()}")

---
## Step 7 — Visualization (Portfolio Figure)

In [ ]:
# Cell 7.1 — Three-panel portfolio figure
import pandas as pd
from sklearn.decomposition import PCA

fig = plt.figure(figsize=(16, 5))
ax1 = fig.add_subplot(1, 3, 1)
ax2 = fig.add_subplot(1, 3, 2)
ax3 = fig.add_subplot(1, 3, 3)

# Panel 1: PCA
pca = PCA(n_components=2)
Z = pca.fit_transform(X_log.T)
evr = pca.explained_variance_ratio_
for label, color in [("tumour", "tomato"), ("normal", "steelblue")]:
    mask = [l == label for l in labels]
    ax1.scatter(Z[mask, 0], Z[mask, 1], c=color, s=60, alpha=0.85,
                edgecolors="white", label=label)
ax1.set_xlabel(f"PC1 ({evr[0]:.1%})")
ax1.set_ylabel(f"PC2 ({evr[1]:.1%})")
ax1.set_title("PCA")
ax1.legend(frameon=False)

# Panel 2: Volcano
neg_log_p = -np.log10(np.clip(pvals, 1e-300, 1))
sig = (np.abs(lfc) > 1) & (pvals < 0.05)
ax2.scatter(lfc[~sig], neg_log_p[~sig], s=6, alpha=0.3, c="lightgray")
ax2.scatter(lfc[sig],  neg_log_p[sig],  s=10, alpha=0.8,
            c=["tomato" if l > 0 else "steelblue" for l in lfc[sig]])
ax2.axvline(-1, color="gray", lw=0.8, ls="--")
ax2.axvline( 1, color="gray", lw=0.8, ls="--")
ax2.axhline(-np.log10(0.05), color="gray", lw=0.8, ls="--")
ax2.set_xlabel("log₂ Fold Change")
ax2.set_ylabel("-log₁₀(p-value)")
ax2.set_title(f"Volcano  ({sig.sum()} DE genes)")

# Panel 3: Heatmap (top 20 DE)
top20 = np.argsort(pvals)[:20]
import seaborn as sns
df_heat = pd.DataFrame(
    X_log[top20],
    index=[gene_ids[i] for i in top20],
    columns=labels,
)
sns.heatmap(df_heat.apply(lambda r: (r - r.mean()) / (r.std() + 1e-9), axis=1),
            cmap="RdBu_r", ax=ax3, yticklabels=True, xticklabels=True, cbar=True)
ax3.set_title("Top 20 DE genes")
ax3.tick_params(axis="y", labelsize=7)
ax3.tick_params(axis="x", rotation=30, labelsize=8)

plt.suptitle("Module 01 Expression Analysis Pipeline — compbio_utils v0", fontsize=12)
plt.tight_layout()

# Save to portfolio
portfolio_dir = repo_root / "portfolio"
portfolio_dir.mkdir(exist_ok=True)
out_path = portfolio_dir / "module01_expression_overview.png"
plt.savefig(out_path, dpi=150, bbox_inches="tight")
print(f"Saved: {out_path}")
plt.show()

---
## Step 8 — Exercises

This is the mini-project — the exercises are the deliverables:

1. **Complete the audit** (Cell 6.1): all functions present in the correct files.
2. **All tests pass** (Cell 6.2): `pytest` exit code 0.
3. **Ruff is clean** (Cell 6.3): no warnings.
4. **Portfolio figure saved** (Cell 7.1): `portfolio/module01_expression_overview.png`.
5. **Write `compbio_utils/README.md`** with:
   - one-paragraph description
   - install instructions
   - quick-start code snippet
   - list of all public functions
6. **Tag `v0.1.0`** in git once all the above pass.

---
## Quiz — Active Recall

1. Why is CPM normalization applied before log2 transformation, not after?
2. The pipeline z-scores columns, not rows. What does that mean, and when would
   you z-score rows instead?
3. Without looking at the code: what does `np.argsort(pvals)[:20]` return?
4. Name all four `compbio_utils` submodules and one function from each.
5. What makes this package a Track B portfolio artifact rather than just working code?

---
## Reflection

**Date completed:** ____________________

> *[Did the full pipeline run using only compbio_utils? Is the portfolio figure saved? What gaps remained after the audit in Cell 6.1?]*

---
*Next: `19_paper_reading_session.ipynb`*